<a href="https://colab.research.google.com/github/zelal-Eizaldeen/deeplearning_course/blob/main/7_2tensorflow_encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Encoder-Decoder Network


- In this programming example, we will **implement an encoder-decoder network** for **neural machine translation** using TensorFlow.

This is a fairly complicated example compared to what we have seen before. So you will probably have to spend some time looking at the code and reading the code to truly understand this.

We start with importing a number of modules that we're going to use.

Note that we are importing **RMSprop** which is a different **optimizer than Adam**, which **turns out that it works better in this case**,  

In [1]:
import numpy as np
import random
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.text \
    import text_to_word_sequence
from tensorflow.keras.preprocessing.sequence \
    import pad_sequences
import tensorflow as tf
import logging
tf.get_logger().setLevel(logging.ERROR)


Then we define a number of **constants**.
- We want to train for **20 epics**.
- We have a **batch size of 128**. We **have a max word vocabulary of 10,000**. - We will limit ourselves to **reading 60,000 examples** from our training file. The reason for that is just to **limit the amount of memory usage **that we need.
- Then we have various constants for the **size of the** **different layers**, the **embeddings**, how **much we want to use for the test data** set and so on.



here is a function that we use to read that file,

The Anki bilingual sentence pairs can be downloaded from http://www.manythings.org/anki/ara-eng.zip.
Unzip it after download and copy the file **ara.txt** to the data directory.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
# Constants
EPOCHS = 20
BATCH_SIZE = 128
MAX_WORDS = 10000
READ_LINES = 60000
LAYER_SIZE = 256
EMBEDDING_WIDTH = 128
TEST_PERCENT = 0.2
SAMPLE_SIZE = 20
OOV_WORD = 'UNK'
PAD_INDEX = 0
OOV_INDEX = 1
START_INDEX = MAX_WORDS - 2
STOP_INDEX = MAX_WORDS - 1
MAX_LENGTH = 60
SRC_DEST_FILE_NAME = '../data/ara.txt'

So we'll do that and we'll **have the training data will be in ara.txt file** that contains **a sentence in Arabic and a corresponding sentence in English**.

We have function** that we use to read that file**, then **split it into a pair of**, the **source language** and the **corresponding destination language sentence**, so we return **src_word_sequences and dest_word_sequences**

In [7]:
# Function to read file.
def read_file_combined(file_name, max_len):
    file = open(file_name, 'r', encoding='utf-8')
    src_word_sequences = []
    dest_word_sequences = []
    for i, line in enumerate(file):
        if i == READ_LINES:
            break
        pair = line.split('\t')
        word_sequence = text_to_word_sequence(pair[1])
        src_word_sequence = word_sequence[0:max_len]
        src_word_sequences.append(src_word_sequence)
        word_sequence = text_to_word_sequence(pair[0])
        dest_word_sequence = word_sequence[0:max_len]
        dest_word_sequences.append(dest_word_sequence)
    file.close()
    return src_word_sequences, dest_word_sequences

 We also need **to have functions for tokenizing and untokenizing words.** So we are declaring this tokenize that we will then use so that we can **transform a sequence of word into indices and then we can go in the other direction as well.** We can provide a token and we will get words back.

 we are going to use **START and STOP** tokens. So we want this network to tell **us ourselves when it's done translating a sentence.**

 So we're basically going to provide it with a sentence in one language and then we give it a **START** token, and then that **START** token is the indication that it's supposed to start **translating and provide the translation of that sentence**, but in a different language. And then it's going to **start emitting a number of words until it's done with that sentence and it will indicate that it's done with that sentence** by emitting a **STOP** token. So we will create our training so that the sequence start with a START token and then ends with a STOP token, and that will teach the model that it should always emit a STOP token at the end of a sentence. So those are helper functions that we're going to use

In [9]:
# Functions to tokenize and un-tokenize sequences.
def tokenize(sequences):
    # "MAX_WORDS-2" used to reserve two indices
    # for START and STOP.
    tokenizer = Tokenizer(num_words=MAX_WORDS-2,
                          oov_token=OOV_WORD)
    tokenizer.fit_on_texts(sequences)
    token_sequences = tokenizer.texts_to_sequences(sequences)
    return tokenizer, token_sequences

def tokens_to_words(tokenizer, seq):
    word_seq = []
    for index in seq:
        if index == PAD_INDEX:
            word_seq.append('PAD')
        elif index == OOV_INDEX:
            word_seq.append(OOV_WORD)
        elif index == START_INDEX:
            word_seq.append('START')
        elif index == STOP_INDEX:
            word_seq.append('STOP')
        else:
            word_seq.append(tokenizer.sequences_to_texts(
                [[index]])[0])
    print(word_seq)

Then we call **read_file_combined**. That will give us the src_seq, so that's in Arabic and then dest_seq which are in English, and then we will tokenize these. So this tokenized method provides both **a tokenized itself that we're gonna use later and then also the tokenized sequences**. Then we do another tokenize,This is in English. We're going to tokenize the destination sequences. So now we have a destination tokenize and here are the tokenized sentences in English.

In [10]:
# Read file and tokenize.
src_seq, dest_seq = read_file_combined(SRC_DEST_FILE_NAME,
                                       MAX_LENGTH)
src_tokenizer, src_token_seq = tokenize(src_seq)
dest_tokenizer, dest_token_seq = tokenize(dest_seq)

In [23]:
def inspect_tokenizer(tok, name="tokenizer"):
    print(f"\n=== {name} ===")
    print(type(tok))
    # Keras Tokenizer attributes
    if hasattr(tok, "word_index"):
        print(f"vocab size (keras): {len(tok.word_index):,}")
        # show a few entries
        ex = list(tok.word_index.items())[:10]
        print("word_index sample:", ex)
    # Hugging Face tokenizer attributes
    if hasattr(tok, "vocab_size"):
        print(f"vocab size (HF): {tok.vocab_size:,}")
    if hasattr(tok, "pad_token"):
        print("pad_token:", getattr(tok, "pad_token", None))
    if hasattr(tok, "bos_token"):
        print("bos/eos:", getattr(tok, "bos_token", None), getattr(tok, "eos_token", None))
def inspect_sequences(seqs, name="seqs", n_preview=3, n_tokens=20):
    import numpy as np
    print(f"\n=== {name} ===")
    print("type:", type(seqs))
    try:
        arr = np.array(seqs, dtype=object)
        print("num sequences:", len(seqs))
        lens = [len(s) for s in seqs]
        if lens:
            print(f"len(min/mean/max): {min(lens)} / {sum(lens)/len(lens):.1f} / {max(lens)}")
    except Exception as e:
        print("length:", len(seqs), "| (could not summarize lengths:", e, ")")
    # preview a few
    for i, s in enumerate(seqs[:n_preview]):
        print(f"[{i}] len={len(s)} ids[:{n_tokens}] ->", s[:n_tokens])

def try_decode(tok, seqs, name="decoded", n_preview=2, n_chars=120):
    print(f"\n=== {name} (decode preview) ===")
    if hasattr(tok, "decode"):
        # HF-style
        for i, s in enumerate(seqs[:n_preview]):
            text = tok.decode(s, skip_special_tokens=True)
            print(f"[{i}] {text[:n_chars]!r}")
    elif hasattr(tok, "index_word"):
        # Keras-style inverse map
        inv = tok.index_word
        for i, s in enumerate(seqs[:n_preview]):
            text = " ".join(inv.get(t, f"<{t}>") for t in s)[:n_chars]
            print(f"[{i}] {text!r}")
    else:
        print("No decode method/index_word available.")

In [24]:
dest_tokenizer, dest_token_seq = tokenize(dest_seq)

inspect_tokenizer(src_tokenizer, "src_tokenizer")
inspect_sequences(src_token_seq, "src_token_seq")
try_decode(src_tokenizer, src_token_seq, "src decode")

inspect_tokenizer(dest_tokenizer, "dest_tokenizer")
inspect_sequences(dest_token_seq, "dest_token_seq")
try_decode(dest_tokenizer, dest_token_seq, "dest decode")


=== src_tokenizer ===
<class 'keras.src.legacy.preprocessing.text.Tokenizer'>
vocab size (keras): 13,621
word_index sample: [('UNK', 1), ('توم', 2), ('من', 3), ('أن', 4), ('لا', 5), ('في', 6), ('هل', 7), ('ما', 8), ('أنا', 9), ('إلى', 10)]

=== src_token_seq ===
type: <class 'list'>
num sequences: 12569
len(min/mean/max): 1 / 4.4 / 36
[0] len=1 ids[:20] -> [5093]
[1] len=1 ids[:20] -> [3072]
[2] len=2 ids[:20] -> [5094, 950]

=== src decode (decode preview) ===
[0] 'مرحبًا'
[1] 'اركض'

=== dest_tokenizer ===
<class 'keras.src.legacy.preprocessing.text.Tokenizer'>
vocab size (keras): 4,424
word_index sample: [('UNK', 1), ('i', 2), ('you', 3), ('the', 4), ('to', 5), ('a', 6), ('is', 7), ('tom', 8), ('he', 9), ('do', 10)]

=== dest_token_seq ===
type: <class 'list'>
num sequences: 12569
len(min/mean/max): 1 / 5.7 / 34
[0] len=1 ids[:20] -> [897]
[1] len=1 ids[:20] -> [352]
[2] len=1 ids[:20] -> [1598]

=== dest decode (decode preview) ===
[0] 'hi'
[1] 'run'


Then we need to **prepare the training data** in a format that **works for this model**. So if we start with what we want the decoder to actually produce, the dest_target_token_seq for the destination  so we have the dest_token_seq (that is a sentence in English). So we do for x in that, so we're going to basically list all the **words in a sentence and then we append this STOP_INDEX to the end**. So that's how we're going to teach this model to **not only produce the translation**, but also produce then the **STOP_INDEX** so that we know when it's done producing this sentence.



In [ ]:
# Prepare training data.
dest_target_token_seq = [x + [STOP_INDEX] for x in dest_token_seq]

In terms of inputs to this decoder, we want it to get kind of the same **inputs but shifted in time**, So at the first time steps, so we will have this decoder, which will be initialized **with a state, the intermediate state from the** **encoder**, and then we will give it a START token. So that's the first time step. The input it gets is a START token and then we want it to produce the first word in English, and then we're going to take that word and feed it back using autoregression. So this is just like a language model. So we will actually **feed this destination with the English sentence, but shifted in time by one compared to the output.** So that's why we see here on the input to the decoder, we will also look through this **dest_target_token_seq**. So that's the same sequence as we had just prepared here, but we'll also **prepend it with the START index**.

In [14]:
# Prepare training data.
dest_input_token_seq = [[START_INDEX] + x for x in
                        dest_target_token_seq]


And for the src_input_data, so this is what we're providing to the **encoder**. So this is going to be the Arabic, the sentence in Arabic, and here we are just going to pad that so that they all have the same length. So this is going to be our src_input_data. We don't need to do anything special there and then we take the thing that we just had prepared up, the **dest_input_token_seq**. We're also padding that so that they also, all the sentences have the same length. Padding **post** here means that we are **adding zeros to the end of the sequence.**

and then for the **outputs**, we're going to pad it at the end. And we're doing the same thing with the dest_target_data. We're also doing a **padding** So all of this is **to prepare the training data**

In [15]:
src_input_data = pad_sequences(src_token_seq, padding = 'post')
dest_input_data = pad_sequences(dest_input_token_seq,
                                padding='post')
dest_target_data = pad_sequences(
    dest_target_token_seq, padding='post', maxlen
    = len(dest_input_data[0]))

so then once we have these training data, we want to split it into the **training and test set**. So we will do that here.

For the **test set**, we take **total number of rows here times that test percent and then we are randomly pulling a number of examples to become our test indices**. and other remaining ones are **training data**.

In [26]:
# Split into training and test set.
rows = len(src_input_data[:,0])
all_indices = list(range(rows))
test_rows = int(rows * TEST_PERCENT)
test_indices = random.sample(all_indices, test_rows)
train_indices = [x for x in all_indices if x not in test_indices]



And we then actually **pull out those from the arrays with this**, so we have the training indices. So this is for the source destination, both the input and the target data. So now we've created that as for the training data set. We have the remaining one for the test data set.

In [27]:
train_src_input_data = src_input_data[train_indices]
train_dest_input_data = dest_input_data[train_indices]
train_dest_target_data = dest_target_data[train_indices]

test_src_input_data = src_input_data[test_indices]
test_dest_input_data = dest_input_data[test_indices]
test_dest_target_data = dest_target_data[test_indices]



Now another thing we are going to do is in order to get a **feeling for how well this translator is working**, we want **to inspect some examples manually**. So we are going to **create a smaller list of examples**. We're going to take **a subset of the test examples and then later actually print out the translations**. we define **SAMPLE_SIZE is set to be 20**
so we're going to take 20 example from our test data set and inspect in more detail. We are just storing them in these variable sample indices.

In [28]:
# Create a sample of the test set that we will inspect in detail.
test_indices = list(range(test_rows))
sample_indices = random.sample(test_indices, SAMPLE_SIZE)
sample_input_data = test_src_input_data[sample_indices]
sample_target_data = test_dest_target_data[sample_indices]

 So all of this is kind of **data preparation**. Has nothing to do with the model itself, but it's a big part of the work is to make sure that we have training data and test data in the right formats.

# Model Building - Keras Functional API

So now we're going to move on to **building the model**

So in the past we have used **the Keras Sequential API** because all **of the models we have done has simply been stacking a number of layers on top of each other**. But now that we're building this **encoder-decoder model, it's a little bit more complex topology and we can no longer do that using the Sequential API**.

So we need to do something called **the Keras Functional API**

Using this API, **we declare the layers separately and then we have a separate step where we connect them together.** In the sequential API, we both declared them and connected them at the same time because they were basically, you always connected a layer to the one that you had declared before. So it was just implicit connecting them after each other, but that's not the case with a functional API.

So what we **start with is to declare all the layers we want and we need to also explicitly declare the inputs**. So we have the **enc_embedding_input** and we'll actually be starting with just building the encoder model here. So it's going to have an **input** which is the embedding that we're presenting to it and then we are going to have an **embedding layer**, and its **output dimension will be the width of the embedding and the input dimension will be the maximum number of words in the vocabulary**. And we are going to have **one encoder layer. The first encode layer is an LSTM layer** we want that one to be **return sequences equals true.** **We also want it to have return state equals true** so it will **not only output of the layer, but it'll also output the internal cell state**.

And then we **have a second layer**. **So our encoder model is an embedding layer followed by two LSTM layers.**

In [30]:
# Build encoder model.
# Input is input sequence in source language.
enc_embedding_input = Input(shape=(None, ))

# Create the encoder layers.
enc_embedding_layer = Embedding(
    output_dim=EMBEDDING_WIDTH, input_dim
    = MAX_WORDS, mask_zero=True)
enc_layer1 = LSTM(LAYER_SIZE, return_state=True,
                  return_sequences=True)
enc_layer2 = LSTM(LAYER_SIZE, return_state=True)





But now we need to connect these layers together. So we see here we **take the embedding layer (enc_embedding_layer)  and call that with the inputs to the embedding (enc_embedding_input), so that's how we connect this input layer to the embedding layer**. And the output of that is enc_embedding_layer_outputs and we are going to then use those outputs as input to the to the first LSTM layer, And we see that this call to the first LSTM layer (enc_layer1) will produce a number of outputs here. It'll produce the regular outputs (enc_layer1_outputs) , the h state (enc_layer1_state_h) and the c state.

So then we are calling the next LSTM layer and it'll take the outputs of this first layer, and that feeds that to the second layer. So this is how we connect the different layers together into a model.

In [31]:
# Connect the encoder layers.
# We don't use the last layer output, only the state.
enc_embedding_layer_outputs = \
    enc_embedding_layer(enc_embedding_input)
enc_layer1_outputs, enc_layer1_state_h, enc_layer1_state_c = \
    enc_layer1(enc_embedding_layer_outputs)
_, enc_layer2_state_h, enc_layer2_state_c = \
    enc_layer2(enc_layer1_outputs)

And once we have done that, we're going to **create an actual model from this**, and we do that by stating that we want to build a model and that's followed by **two arguments. One is the inputs to the model which is the encoder embedding input and the next argument is the output from this model, and the output from this model is a list of the following variables.**

So we see that it will **output the h state and the c state for the first LSTM layer, as well as the second LSTM layer.** And we need that as output because that's what we're going to use then **as input to the decoder model.** So this is our entire encoder model and we can then print that one out.

In [32]:
# Build the model.
enc_model = Model(enc_embedding_input,
                  [enc_layer1_state_h, enc_layer1_state_c,
                   enc_layer2_state_h, enc_layer2_state_c])
enc_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │  1,280,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, None,     │    394,240 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ not_equal_1[0][0] │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, 256),     │    525,312 │ lstm_2[0][0],     │
│                     │ (None, 256),      │            │ not_equal_1[0][0] │
│                     │ (None, 256)]      │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,199,552 (8.39 MB)

 Trainable params: 2,199,552 (8.39 MB)

 Non-trainable params: 0 (0.00 B)

So here we see that we have our **input layer, we have our embedding layer, and then we have one LSTM and the second LSTM,**



#Decoder

So then it's time to build our decoder model, which will have **more inputs because it will not only have the embedding input, but it will also have the initial internal state.**

So we will have the h and c input for the first LSTM layer, as well as the h and c inputs to the second LSTM layer.



In [33]:
# Build decoder model.
# Input to the network is input sequence in destination
# language and intermediate state.
dec_layer1_state_input_h = Input(shape=(LAYER_SIZE,))
dec_layer1_state_input_c = Input(shape=(LAYER_SIZE,))
dec_layer2_state_input_h = Input(shape=(LAYER_SIZE,))
dec_layer2_state_input_c = Input(shape=(LAYER_SIZE,))
dec_embedding_input = Input(shape=(None, ))

So here we create these decoder layers. We create the embedding layer. **We create the LSTM layers and then the decoder model will finally consist of also a fully connected layer with soft max on the top.**

So the encoder was just embedding and then two LSTM layers. The decoder is an embedding layer, two LSDM layers, and a fully connected layer.

And again, **we're using the functional API here**, so this is just to **declare the layers**. We also need to then connect them together and that's what we're doing here next.

In [34]:
# Create the decoder layers.
dec_embedding_layer = Embedding(output_dim=EMBEDDING_WIDTH,
                                input_dim=MAX_WORDS,
                                mask_zero=True)
dec_layer1 = LSTM(LAYER_SIZE, return_state = True,
                  return_sequences=True)
dec_layer2 = LSTM(LAYER_SIZE, return_state = True,
                  return_sequences=True)
dec_layer3 = Dense(MAX_WORDS, activation='softmax')

So for the decoder embedding layer, we connect that to its input and then we take the output of this embedding layer and we provide that to the input of the first LSTM layer.


But here we do something else as well.** We say that the initial state for this layer should be taken from the h and c inputs.** So this is **how it transfers the state from the encoder model to the decoder model during the first time step**.

In [36]:
# Connect the decoder layers.
dec_embedding_layer_outputs = dec_embedding_layer(
    dec_embedding_input)
dec_layer1_outputs, dec_layer1_state_h, dec_layer1_state_c = \
    dec_layer1(dec_embedding_layer_outputs,
    initial_state=[dec_layer1_state_input_h,
                   dec_layer1_state_input_c])


We do the same thing for **the decoder layer two, so that's the second LSTM layer**. It's going to get its input from the output of the previous layer and again, it's initial state should be taken from the encoder. So we're going to supply this explicitly as inputs to the model. And then finally we have our output layer, which takes the output of the second LSTM layer as input and produces this output.

In [37]:
dec_layer2_outputs, dec_layer2_state_h, dec_layer2_state_c = \
    dec_layer2(dec_layer1_outputs,
    initial_state=[dec_layer2_state_input_h,
                   dec_layer2_state_input_c])
dec_layer3_outputs = dec_layer3(dec_layer2_outputs)

# Model Building

Looking at this model, this model takes as the input. Both the **decoder embedding input, as well as the initial state here that was supplied from the encode**r.

But then it also produces as outputs, Not only the output of the **top layer which is decoder layer3 output. It also produces the h and c state for both layers, so this is going to be used in autoregression later**. So we're going to call this model. **We're going to supply it with a START token, as well as the input state from the encoder** and that's going to **produce the first predicted word**.

 We are going to then, **that word as well as the output state, to feed that back in the next time step.** We're going to **take that state as input to the model and the predicted word as input to the embedding, and that's going to give us the next one.**

 And we're going to do that in **an autoregressive manner until the model predicts a STOP token.** That's when it's done translating.

In [38]:
# Build the model.
dec_model = Model([dec_embedding_input,
                   dec_layer1_state_input_h,
                   dec_layer1_state_input_c,
                   dec_layer2_state_input_h,
                   dec_layer2_state_input_c],
                  [dec_layer3_outputs, dec_layer1_state_h,
                   dec_layer1_state_c, dec_layer2_state_h,
                   dec_layer2_state_c])
dec_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 128) │  1,280,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, None,     │    394,240 │ embedding_2[1][0… │
│                     │ 256), (None,      │            │ input_layer_2[0]… │
│                     │ 256), (None,      │            │ input_layer_3[0]… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_4       │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, None,     │    525,312 │ lstm_4[1][0],     │
│                     │ 256), (None,      │            │ input_layer_4[0]… │
│                     │ 256), (None,      │            │ input_layer_5[0]… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  2,570,000 │ lstm_5[1][0]      │
│                     │ 10000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,769,552 (18.19 MB)

 Trainable params: 4,769,552 (18.19 MB)

 Non-trainable params: 0 (0.00 B)

So we do this and print out the summary for this model and we see, we have **the inputs, including input layer, the embedding layer, other input layers that are the states.** **We have an LSTM**, **the input layers to the other LSTM which is the state for them**, **We have that LSTM and then we have the final output layer**.



# Training Process

**So now we have an encoder model. We have a decoder model.**

We want to **combine them into the full model** that we can use for **training**.

So we have the encoder embedding input, the decoder embedding input for this training model,

In [39]:
# Build and compile full training model.
# We do not use the state output when training.
train_enc_embedding_input = Input(shape=(None, ))
train_dec_embedding_input = Input(shape=(None, ))

and then we are going to get an **intermediate state which is coming from the encoder model**.

In [40]:
intermediate_state = enc_model(train_enc_embedding_input)

And here is how we **are now connecting things together. We're saying that the decoder model will, as its input, have the training decoder embedding input** **and it will also receive the intermediate state as inputs**.

In [41]:
train_dec_output, _, _, _, _ = dec_model(
    [train_dec_embedding_input] +
    intermediate_state)

then from that, we **can build the final training model**. And the training model has two inputs, so it's the embedding for the encoder, the embedding for the decoder, and **then it's producing a output from the decoder**.

In [42]:
training_model = Model([train_enc_embedding_input,
                        train_dec_embedding_input],
                        train_dec_output)

And as an optimizer, we want to use **RMSprop**.

In [43]:
optimizer = RMSprop(learning_rate=0.01)


We want to use something known as sparse categorical cross entropy this time. **So it's technically the same thing as a categorical cross entropy, but we don't need to actually describe the outputs as one-hot encoded**. We will instead **describe this as the index that we are trying to predict. This is really just a memory optimization,** so we don't have to have as big training data because when you do a one-hot encoding, **you're just going to have a lot of zeros**. And if you have a fairly big vocabulary and a lot of training examples, that's going to take up a lot of memory. **So using this sparse category, we'll cross entropy. We will reduce the need for memory in the model**,  

In [44]:
training_model.compile(loss='sparse_categorical_crossentropy',
                       optimizer=optimizer, metrics =['accuracy'])

We print out the summary of the model,

In [45]:
training_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ [(None, 256),     │  2,199,552 │ input_layer_7[0]… │
│ (Functional)        │ (None, 256),      │            │                   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_2        │ [(None, None,     │  4,769,552 │ input_layer_8[0]… │
│ (Functional)        │ 10000), (None,    │            │ functional_1[0][… │
│                     │ 256), (None,      │            │ functional_1[0][… │
│                     │ 256), (None,      │            │ functional_1[0][… │
│                     │ 256), (None,      │            │ functional_1[0][… │
│                     │ 256)]             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,969,104 (26.59 MB)

 Trainable params: 6,969,104 (26.59 MB)

 Non-trainable params: 0 (0.00 B)

and we see that we have our model, with **these two input layers and then we have our encoder model and our decoder model.**

# Testing

And now **we are ready to train and test this model**. So in this case.

 What we will do is that we w**ill train it for one epic and then we will inspect these 20 examples** **to see how it results, and then we'll do another training for one epic.**

 - The **outer loop that will run for a number of epics** and then when we call the fit function, we will just say train for one epic and then we will call this f**it function repeatedly**.

 So **this training model** has **fit** method where   We say that the **input here is now a list which has both the source input data and the destination input data**. So this is both the Arabic and the English sentences and **then we have the destination data**, **which is the output**. And the same thing on the validation side. **We have a list as input and then the output side**  and **the batch size** So this will **train the model for one epic**.

 - ** First Inner loop**: What we are then going to use this model on some of these test examples **and see what translations it will produce**, and that's also a somewhat manual process where we're going to use **autoregression**. So in this first inner loop, We are **looping through these 20 samples**. So we are taking **out one sample_input_data, sample_target_data**. So this is basically one Arabic sentence and one English sentence. And then we're **taking the Arabic sentence and run that through the encoder model**. So here when we do this **autoregression**, we are not using the joint model. We're going to use **the split encoder model** **and decoder model separately**. So we start running it through the encoder model (**enc_model.predict**) to get the intermediate state (**last_states**) that one produces. That's all that we need from the encoder model.

 And now we can **start doing autoregression with a decoder model**. we're going to **present the decoder model with this state**(last_states). **We're going to initialize it to that state** and to give it a **START_INDEX**, which is the start token, that tells the **intermediate state** **to start translating into English**

 So it's going to **have the intermediate state will describe what is the overall meaning of the sentence and then it's going to generate the first word in English for that sentence**. **Then we're going to take that word, feed it back in the next time step, and it's going to generate the next one and we'll do that until it predicts a STOP token**. It might not actually predict a STOP token, so **then we will stop automatically** if we have predicted too many words. So if we do more than max length, we will automatically stop
  **range(MAX_LENGTH)**
- **So in the second for loop**, we'll have a for loop that will break out or the for loop if it finds the STOP token, and otherwise it'll stop after max length number of words.

So here we can see how we **use the decoder model** (**dec_model.predict**)  and we do a prediction and the **output from this model will be these are the predicted words and it will also output its internal states.** And those **internal states we're gonna later use as the input to the model again 'cause we need to initialize this** **with the next set of states that it has produced**.

 So we see the call **dec_model.predict** to the decoder model. We take a list consisting of the input word **[x]**and then we **concatenate that with these states**. The last states is a list of **these four state variables**.

 So each time we **call predict**, we **read out the states and we get a prediction**. And then we do the **argmax** to **find what is the actual word that it has the highest probability?** And then we put that into  **predicted_word_sequence**, so that's what we're going to use to print it out later.



In [ ]:
# Train and test repeatedly.
for i in range(EPOCHS):
    print('step: ' , i)
    # Train model for one epoch.
    history = training_model.fit(
        [train_src_input_data, train_dest_input_data],
        train_dest_target_data, validation_data=(
            [test_src_input_data, test_dest_input_data],
            test_dest_target_data), batch_size=BATCH_SIZE,
        epochs=1)

    # Loop through samples to see result
    for (test_input, test_target) in zip(sample_input_data,
                                         sample_target_data):
        # Run a single sentence through encoder model.
        x = np.reshape(test_input, (1, -1))
        last_states = enc_model.predict(
            x, verbose=0)
        # Provide resulting state and START_INDEX as input
        # to decoder model.
        prev_word_index = START_INDEX
        produced_string = ''
        pred_seq = []
        for j in range(MAX_LENGTH):
            x = np.reshape(np.array(prev_word_index), (1, 1))
            # Predict next word and capture internal state.
            preds, dec_layer1_state_h, dec_layer1_state_c, \
                dec_layer2_state_h, dec_layer2_state_c = \
                    dec_model.predict(
                        [x] + last_states, verbose=0)
            last_states = [dec_layer1_state_h,
                           dec_layer1_state_c,
                           dec_layer2_state_h,
                           dec_layer2_state_c]
            # Find the most probable word.
            prev_word_index = np.asarray(preds[0][0]).argmax()
            pred_seq.append(prev_word_index)
            if prev_word_index == STOP_INDEX:
                break
        tokens_to_words(src_tokenizer, test_input)
        tokens_to_words(dest_tokenizer, test_target)
        tokens_to_words(dest_tokenizer, pred_seq)
        print('\n\n')

step:  0
79/79 ━━━━━━━━━━━━━━━━━━━━ 664s 8s/step - accuracy: 0.4838 - loss: 6.4841 - val_accuracy: 0.0628 - val_loss: 4.7073
['هذه', 'أكبر', 'قطة', 'رأيتها', 'في', 'حياتي', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
['this', 'is', 'the', 'biggest', 'cat', 'that', "i've", 'ever', 'seen', 'STOP', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
['i', 'you', 'to', 'to', 'STOP']



['خَطهُ', 'رديء', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
['his', 'handwriting', 'is', 'poor', 'STOP', 'PAD', 'PAD', 'PAD

But we also use this **previous_word_index at the next time around in our loop where** we're going to use that as the **input in the next time step**. So this will basically come up with a translation

In [ ]:
for j in range(MAX_LENGTH):
            x = np.reshape(np.array(prev_word_index), (1, 1))
            # Predict next word and capture internal state.
            preds, dec_layer1_state_h, dec_layer1_state_c, \
                dec_layer2_state_h, dec_layer2_state_c = \
                    dec_model.predict(
                        [x] + last_states, verbose=0)
            last_states = [dec_layer1_state_h,
                           dec_layer1_state_c,
                           dec_layer2_state_h,
                           dec_layer2_state_c]
            # Find the most probable word.
            prev_word_index = np.asarray(preds[0][0]).argmax()
            pred_seq.append(prev_word_index)

and then we can take **these inputs and what the targets should be, as well as the predicted targets**. And we do convert that to words, so we can then print that out. So each of this will print out the corresponding sentence.

In [ ]:
tokens_to_words(src_tokenizer, test_input)
tokens_to_words(dest_tokenizer, test_target)
tokens_to_words(dest_tokenizer, pred_seq)
print('\n\n')

This is the Arabic sentence,

In [ ]:
tokens_to_words(src_tokenizer, test_input)


this is the true English sentence

In [ ]:
tokens_to_words(dest_tokenizer, test_target)


and then this is the sentence that the model created.

In [ ]:
tokens_to_words(dest_tokenizer, pred_seq)


so now we can see after # of epic and we can see here is the input sentence in Arabic, each **word by word and the output was ...**.

 We got the printout of these 20 sample sentences and saw over time how t**hose translations get better and better.**

 So it was not doing so well here after just one epic, we can look at a couple of more examples.




After training our 20 epics, we can take a look at the **translations** that the network provides and compare that to what the actual translations are.

 So we can see here that with a relatively simply code, we can produce a network that can do some translation here between Arabic and English. That concludes the neural machine translation example with Tensorflow.